# Sheet 4.4: Monitoring, drift and the capstone

**Roadmap topic 4.4** (Fanout 11.18-11.20, 11.25). Read first: Prometheus *Getting started* and the docs on metric types and histograms, Grafana *Getting started*, MLOps Zoomcamp module 5, the Evidently README.

Exercises 1-4 have auto-checks (CPU, `pip install fastapi httpx prometheus_client pyyaml scipy`). Exercise 5 is the capstone that closes the roadmap.

**Rules:** fill every `# YOUR CODE` cell, never edit `CHECK` cells.

In [ ]:
import time, math, numpy as np, yaml
from scipy import stats
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient
from prometheus_client import CollectorRegistry, Counter, Histogram, Gauge, generate_latest, CONTENT_TYPE_LATEST
from starlette.responses import Response

## Exercise 1: Instrument an API with Prometheus metrics

`make_monitored_app(registry)` returns a FastAPI app with:
- `GET /predict?x=<float>`: returns `{"y": 2*x}`; if `x < 0` it raises a 500 error (simulate a bug) by returning `Response(status_code=500)`.
- `GET /metrics`: Prometheus text format from `registry`.
- An HTTP **middleware** that records for every request except `/metrics`:
  - `http_requests_total` Counter with labels `path` and `status`
  - `http_request_duration_seconds` Histogram with label `path` and buckets `(0.005, 0.01, 0.05, 0.1, 0.5, 1, 5)`
  - `http_requests_in_flight` Gauge (inc on entry, dec on exit, even on errors)

Register all metrics on the given `registry` (not the global default), so tests are isolated.

In [ ]:
def make_monitored_app(registry):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 1 (do not edit) ----
def _check1():
    reg = CollectorRegistry(); c = TestClient(make_monitored_app(reg), raise_server_exceptions=False)
    for x in (1, 2, 3, -1, 4, -5): c.get(f"/predict?x={x}")
    assert c.get("/predict?x=2").json() == {"y": 4.0}
    txt = c.get("/metrics").text
    get = lambda name, labels: reg.get_sample_value(name, labels)
    assert get("http_requests_total", {"path": "/predict", "status": "200"}) == 5
    assert get("http_requests_total", {"path": "/predict", "status": "500"}) == 2
    assert get("http_request_duration_seconds_count", {"path": "/predict"}) == 7
    assert get("http_request_duration_seconds_bucket", {"path": "/predict", "le": "5.0"}) == 7
    assert get("http_requests_in_flight", {}) == 0, "in-flight gauge must return to 0"
    assert get("http_requests_total", {"path": "/metrics", "status": "200"}) is None, "don't count /metrics itself"
    assert "http_request_duration_seconds_bucket" in txt
    print("PASS")
_check1()

## Exercise 2: Quantiles from histogram buckets

Prometheus can't store raw latencies, so `histogram_quantile()` estimates quantiles from cumulative bucket counts by **linear interpolation inside the bucket** where the quantile falls (lower bound of the first bucket is 0). Implement `histogram_quantile(q, buckets)` where `buckets` is a list of `(upper_bound, cumulative_count)` ending with `(inf, total)`. If the quantile falls in the `+Inf` bucket, return the largest finite bound (as Prometheus does).

Then answer: why can your p99 estimate be badly wrong, and how do you choose buckets?

In [ ]:
def histogram_quantile(q, buckets):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 2 (do not edit) ----
def _check2():
    B = [(0.1, 10), (0.5, 60), (1.0, 90), (2.0, 99), (float("inf"), 100)]
    assert abs(histogram_quantile(0.5, B) - (0.1 + 0.4 * 40 / 50)) < 1e-12
    assert abs(histogram_quantile(0.05, B) - 0.05) < 1e-12
    assert abs(histogram_quantile(0.95, B) - (1.0 + 1.0 * 5 / 9)) < 1e-12
    assert histogram_quantile(0.999, B) == 2.0
    r = np.random.default_rng(0); lat = r.lognormal(np.log(0.15), 0.6, 50_000)
    bounds = [0.005, 0.01, 0.025, 0.05, 0.1, 0.25, 0.5, 1, 2.5, 5, 10, float("inf")]
    Bk = [(b, int((lat <= b).sum())) for b in bounds]
    est, true = histogram_quantile(0.95, Bk), np.quantile(lat, 0.95)
    print(f"PASS  p95 true {true*1000:.0f} ms vs bucket estimate {est*1000:.0f} ms (error {abs(est-true)/true:.0%})")
_check2()

**Written (2a):** Explain the error you just saw. If your SLO is "p95 < 300 ms", where must bucket boundaries be?

**Your answer:**

## Exercise 3: Detecting input drift

Your model was trained on prompts with a certain length distribution; production traffic shifts. Implement:
- `psi(expected, actual, bins=10)`: Population Stability Index with **quantile bins from `expected`** (so each bin holds ~10% of the reference), `eps = 1e-6` to avoid log(0): `Σ (a - e)·ln(a / e)`.
- `ks_statistic(x, y)`: the two-sample Kolmogorov-Smirnov statistic `max |F_x - F_y|`, computed yourself from sorted samples (match `scipy.stats.ks_2samp(...).statistic`).
- `drift_report(ref, cur)`: `dict(psi=..., ks=..., p_value=..., drift=bool)` with `drift = psi > 0.2 or p_value < 0.01` (use scipy for the p-value).

In [ ]:
def psi(expected, actual, bins=10): raise NotImplementedError
def ks_statistic(x, y): raise NotImplementedError
def drift_report(ref, cur): raise NotImplementedError

In [ ]:
# ---- CHECK 3 (do not edit) ----
def _check3():
    r = np.random.default_rng(0)
    ref = r.lognormal(4, 0.6, 20_000); same = r.lognormal(4, 0.6, 5_000); shifted = r.lognormal(4.4, 0.7, 5_000)
    for a, b in ((ref, same), (ref, shifted), (r.normal(size=300), r.normal(0.3, 1, 200))):
        assert abs(ks_statistic(a, b) - stats.ks_2samp(a, b).statistic) < 1e-12, "KS statistic differs from scipy"
    assert psi(ref, same) < 0.02 and psi(ref, shifted) > 0.2, (psi(ref, same), psi(ref, shifted))
    assert not drift_report(ref, same)["drift"] and drift_report(ref, shifted)["drift"]
    print(f"PASS  PSI no-drift {psi(ref, same):.3f}, drifted {psi(ref, shifted):.2f}")
_check3()

## Exercise 4: Alerting rules

Write `RULES`, a Prometheus rule file (YAML) with a group `lm-api` containing two alerts:
- `HighP95Latency`: p95 of `http_request_duration_seconds` over 5-minute windows, aggregated across pods **by `le`**, above 0.3 s for 10 minutes; severity `page`.
- `HighErrorRate`: ratio of 5xx requests to all requests over 5 minutes above 2% for 5 minutes; severity `ticket`.
Each alert needs `annotations.summary`.

In [ ]:
RULES = '''
# YOUR RULES
'''

In [ ]:
# ---- CHECK 4 (do not edit) ----
def _check4():
    g = yaml.safe_load(RULES)["groups"][0]; assert g["name"] == "lm-api"
    rules = {r["alert"]: r for r in g["rules"]}
    lat, err = rules["HighP95Latency"], rules["HighErrorRate"]
    e = lat["expr"].replace(" ", "")
    assert e.startswith("histogram_quantile(0.95,") and "rate(http_request_duration_seconds_bucket[5m])" in e and "by(le)" in e and e.endswith(">0.3")
    assert lat["for"] == "10m" and lat["labels"]["severity"] == "page" and lat["annotations"]["summary"]
    e2 = err["expr"].replace(" ", "")
    assert 'status=~"5.."' in e2 and e2.count("rate(http_requests_total") == 2 and e2.endswith(">0.02")
    assert err["for"] == "5m" and err["labels"]["severity"] == "ticket" and err["annotations"]["summary"]
    print("PASS")
_check4()

**Written (4a):** Why must you aggregate with `sum by (le)` *before* `histogram_quantile`, and never average per-pod p95 values?

---
## Exercise 5 (laptop): monitoring stack and capstone

1. On your `kind` cluster from sheet 4.3: `helm install` **kube-prometheus-stack**, add a `ServiceMonitor` for your API, load your `RULES`, and build a Grafana dashboard with: request rate, error rate, p50/p95/p99 latency (from buckets), in-flight requests, tokens/sec.
2. Inject faults (send negative inputs, add an artificial `sleep`) and show each alert firing and resolving. Screenshot both.
3. Drift: log every prompt's length and a topic label (e.g. from a zero-shot classifier) to a file; run `drift_report` (or Evidently's report) daily against your training data with a GitHub Actions `schedule:` workflow; publish the report as an artifact.
4. **Capstone:** one repository that goes data → train → evaluate → register → build → deploy → monitor, with an architecture diagram (Mermaid in the README is fine), and a README a reviewer can follow from `git clone` to a live Grafana dashboard.

## Final checkpoint
You have four public portfolio pieces: the NumPy MLP (Step 1), your trained small LLM with write-up (Step 2), the GRPO-tuned model with evaluation (Step 3), and the paper plus this production repo (Step 4). Put them at the top of your GitHub profile and CV, and start applying.

---
### Hints for self-grading
- 2a: inside a bucket the estimator assumes latencies are uniformly spread, so with wide buckets (here 0.25-0.5 s) the estimate can be off by a lot. Put boundaries densely around the values your SLO cares about (e.g. 0.2, 0.25, 0.3, 0.35 s).
- 4a: quantiles don't average: the mean of per-pod p95s is not the p95 of all requests. Summing bucket rates across pods first gives the true combined histogram.